# Agent Evaluation Metrics with DeepEval

Demo notebook for https://deepeval.com/guides/guides-ai-agent-evaluation-metrics

We build one deliberately simple travel-booking agent (LangChain `create_agent` +
LangGraph under the hood) and use it to demonstrate DeepEval's six agent evaluation
metrics across the three layers DeepEval describes:

- **Reasoning layer:** `PlanQualityMetric`, `PlanAdherenceMetric`
- **Action layer:** `ToolCorrectnessMetric`, `ArgumentCorrectnessMetric`
- **Execution layer:** `TaskCompletionMetric`, `StepEfficiencyMetric`

The agent itself is intentionally trivial — three fake tools, no memory, no
middleware — all the interesting part of this notebook is the *evaluation*, not
the agent.

In [2]:
%pip install -q deepeval langchain langchain-groq langgraph python-dotenv

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.24.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup — same model as previous labs (Groq via `langchain`)

In [1]:
import os
import random
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in your .env file")

MODEL = "groq:openai/gpt-oss-20b"  # same model used in previous lab notebooks

print("Ready. Model:", MODEL)

Ready. Model: groq:openai/gpt-oss-20b


## Judge model

DeepEval's LLM-as-judge metrics default to an OpenAI model. We only have a
`GROQ_API_KEY`, so we wrap the same Groq chat model in a small
`DeepEvalBaseLLM` adapter and pass that in as `model=...` to every metric
instead. This is the pattern DeepEval's own docs recommend for plugging in a
custom/LangChain model.

In [9]:
from langchain.chat_models import init_chat_model
from deepeval.models.base_model import DeepEvalBaseLLM


class GroqJudge(DeepEvalBaseLLM):
    """Lets DeepEval's judge-based metrics run on our existing Groq model/key."""

    def __init__(self, model_name: str = MODEL):
        self.model_name = model_name
        self.chat_model = init_chat_model(model_name, temperature=0)

    def load_model(self):
        return self.chat_model

    def generate(self, prompt: str, schema=None):
        if schema is not None:
            # "json_mode" is more reliable than the tool-calling default on Groq models.
            return self.chat_model.with_structured_output(schema, method="json_mode").invoke(prompt)
        return self.chat_model.invoke(prompt).content

    async def a_generate(self, prompt: str, schema=None):
        if schema is not None:
            structured_llm = self.chat_model.with_structured_output(schema, method="json_mode")
            return await structured_llm.ainvoke(prompt)
        response = await self.chat_model.ainvoke(prompt)
        return response.content

    def get_model_name(self) -> str:
        return self.model_name


judge_model = GroqJudge()

## A deliberately simple agent

Three fake tools (search flights, check weather, book a flight) and one
`create_agent` call. Nothing fancy — the point is to have *something* to
evaluate, not to build an impressive agent.

In [10]:
from langchain.tools import tool
from langchain.agents import create_agent

_FLIGHTS = {
    ("Cairo", "Paris"): [{"id": "FL100", "price": 480}, {"id": "FL101", "price": 390}],
    ("Cairo", "London"): [{"id": "FL200", "price": 520}],
}


@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search available flights between two cities for a given date (YYYY-MM-DD)."""
    options = _FLIGHTS.get((origin, destination), [{"id": "FL999", "price": 450}])
    return str(options)


@tool
def get_weather(city: str) -> str:
    """Get the current (fake) weather for a city."""
    return f"It's sunny and 27C in {city}."


@tool
def book_flight(flight_id: str) -> str:
    """Book a flight by its ID and return a confirmation."""
    return f"Booked {flight_id}. Confirmation: CONF-{random.randint(1000, 9999)}"


travel_agent = create_agent(
    model=MODEL,
    tools=[search_flights, get_weather, book_flight],
    system_prompt=(
        "You are a travel-booking assistant. Search for flights before booking, "
        "and always book the cheapest option unless told otherwise."
    ),
)

# sanity check
test_result = travel_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Cairo?"}]}
)
print(test_result["messages"][-1].content)

The current weather in Cairo is sunny with a temperature of 27 °C.


## A small dataset

A couple of `Golden`s to run the agent against. No ground-truth output is
needed for most of these metrics — DeepEval judges the agent's own trace.

> Note: running `evals_iterator` / `evaluate` without `deepeval login` just
> evaluates and prints locally — no Confident AI account is needed for this
> demo.

In [11]:
from deepeval.dataset import EvaluationDataset, Golden

goldens = [
    Golden(input="Find and book the cheapest flight from Cairo to Paris for 2025-03-15."),
    Golden(input="What's the weather like in Cairo right now?"),
]

dataset = EvaluationDataset(goldens=goldens)

## Reasoning + execution layer: trajectory-level metrics

`PlanQualityMetric`, `PlanAdherenceMetric`, `TaskCompletionMetric` all need the
*whole* ordered trace of a single agent run, so we wrap one call to the agent
with `@observe(type="agent")` and attach whichever metric we're demoing via
`dataset.evals_iterator(metrics=[...])`.

In [12]:
from deepeval.tracing import observe


@observe(type="agent")
def run_travel_agent(user_input: str) -> str:
    result = travel_agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    return result["messages"][-1].content

### 1. `PlanQualityMetric` — is the agent's plan logical, complete, efficient?

In [13]:
from deepeval.metrics import PlanQualityMetric

plan_quality = PlanQualityMetric(threshold=0.7, model=judge_model)

for golden in dataset.evals_iterator(metrics=[plan_quality]):
    run_travel_agent(golden.input)

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                ┃ Average Score           ┃ Pass Rate                                       ┃ Total      │
│ ━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━ │
│  Plan Quality          │ 1.00                    │ 100.00% | passed=2 | failed=0                   │ 2          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=181447;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.33s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### 2. `PlanAdherenceMetric` — does the agent actually follow its own plan?

In [14]:
from deepeval.metrics import PlanAdherenceMetric

plan_adherence = PlanAdherenceMetric(threshold=0.7, model=judge_model)

for golden in dataset.evals_iterator(metrics=[plan_adherence]):
    run_travel_agent(golden.input)

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                   ┃ Average Score          ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Plan Adherence           │ 1.00                   │ 100.00% | passed=2 | failed=0                  │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=1391;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.48s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### 3. `TaskCompletionMetric` — did the agent actually accomplish the task?

In [15]:
from deepeval.metrics import TaskCompletionMetric

task_completion = TaskCompletionMetric(threshold=0.7, model=judge_model)

for golden in dataset.evals_iterator(metrics=[task_completion]):
    run_travel_agent(golden.input)

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m00vgxp9ecb9xqa3ak7j7w6d` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7649, Requested 449. Please try again in 735ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### 4. `StepEfficiencyMetric` — does the agent avoid unnecessary steps?

To make the metric's judgment visible, we run it against a deliberately
*inefficient* variant that repeats the same flight search twice before letting
the real agent take over.

In [ ]:
from deepeval.metrics import StepEfficiencyMetric

step_efficiency = StepEfficiencyMetric(threshold=0.7, model=judge_model)


@observe(type="agent")
def inefficient_travel_agent(user_input: str) -> str:
    # Redundant, back-to-back identical search -- StepEfficiencyMetric should penalize this.
    search_flights.invoke({"origin": "Cairo", "destination": "Paris", "date": "2025-03-15"})
    search_flights.invoke({"origin": "Cairo", "destination": "Paris", "date": "2025-03-15"})
    result = travel_agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    return result["messages"][-1].content


efficiency_dataset = EvaluationDataset(
    goldens=[Golden(input="Find and book the cheapest flight from Cairo to Paris for 2025-03-15.")]
)

for golden in efficiency_dataset.evals_iterator(metrics=[step_efficiency]):
    inefficient_travel_agent(golden.input)

## Action layer: component-level metrics

`ToolCorrectnessMetric` and `ArgumentCorrectnessMetric` look at a single
LLM-with-tools call rather than the full multi-step trace, so instead of going
through `create_agent`'s internal loop we make one direct tool-bound model
call and attach the metric straight to that span with
`@observe(type="llm", metrics=[...])`.

In [ ]:
llm_with_tools = init_chat_model(MODEL).bind_tools([search_flights, get_weather, book_flight])

### 5. `ToolCorrectnessMetric` — did the agent call the right tool(s)?

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import ToolCall
from deepeval.dataset import get_current_golden
from deepeval.tracing import update_current_span

tool_correctness = ToolCorrectnessMetric(threshold=0.7)


@observe(type="llm", metrics=[tool_correctness])
def call_llm_for_tools(user_input: str):
    response = llm_with_tools.invoke(user_input)
    tools_called = [ToolCall(name=call["name"]) for call in response.tool_calls]
    update_current_span(
        input=user_input,
        tools_called=tools_called,
        expected_tools=get_current_golden().expected_tools,
    )
    return response


tool_dataset = EvaluationDataset(
    goldens=[
        Golden(input="What's the weather in Paris?", expected_tools=[ToolCall(name="get_weather")]),
        Golden(
            input="Find flights from Cairo to Paris on 2025-03-15.",
            expected_tools=[ToolCall(name="search_flights")],
        ),
    ]
)

for golden in tool_dataset.evals_iterator(metrics=[tool_correctness]):
    call_llm_for_tools(golden.input)

### 6. `ArgumentCorrectnessMetric` — are the tool call arguments correct for the input?

In [ ]:
from deepeval.metrics import ArgumentCorrectnessMetric

argument_correctness = ArgumentCorrectnessMetric(threshold=0.7, model=judge_model)


@observe(type="llm", metrics=[argument_correctness])
def call_llm_for_arguments(user_input: str):
    response = llm_with_tools.invoke(user_input)
    update_current_span(input=user_input)
    return response


argument_dataset = EvaluationDataset(
    goldens=[Golden(input="Find flights from Cairo to Paris for 2025-03-15.")]
)

for golden in argument_dataset.evals_iterator(metrics=[argument_correctness]):
    call_llm_for_arguments(golden.input)

## Recap — when to reach for which metric

| Agent type | Prioritize |
|---|---|
| Explicit planning agents | `PlanQualityMetric` + `PlanAdherenceMetric` |
| Multi-tool workflows | `ToolCorrectnessMetric` + `ArgumentCorrectnessMetric` |
| Production / cost-sensitive | `StepEfficiencyMetric` |
| Task-critical systems | `TaskCompletionMetric` |

Trajectory-level metrics (`PlanQuality`, `PlanAdherence`, `TaskCompletion`,
`StepEfficiency`) are attached via `dataset.evals_iterator(metrics=[...])`
around an `@observe(type="agent")` call. Component-level metrics
(`ToolCorrectness`, `ArgumentCorrectness`) are attached directly on the
decorator of the specific `@observe(type="llm", metrics=[...])` span that
makes the tool-calling decision.